# Example notebook to calculate new Fisher matrices

This notebook provides an example of how to calculate new Fisher matrices, including the numerical derivatives of the CMB and BAO theory with respect to cosmological parameters. It does this using the `Fisher` class of `hdfisher/fisher.py`.

Below, we import the required code to run this notebook:

In [1]:
import os
from IPython.display import display
import numpy as np
import pandas as pd
from hdfisher import fisher

# Code initialization and setup

There is one __required__ argument when initializing the `Fisher` class:
- `fisher_dir`: The _absolute_ path to the directory that will hold the output. If the directory does not exist, it will be created upon initialization, along with the following files and directories:
  1. `fiducial_params.yaml` and `step_sizes.yaml` : Files holding the fiducial parameter values, and the step sizes used for each varied parameter, respectively. This information is used to calculate the numerical derivatives.
      - We provide our fiducial parameter values (including accuracy parameters) and Fisher step sizes, which are used by default: see `config.fiducial_params` and `config.fiducial_fisher_step_sizes`.
      - See the `fiducial_params` and `step_sizes` arguments below if you would like to use your own. 
  2. `theory`: An empty directory that will be used to store the theory calculation when each parameter is varied.
  3. `derivs`: An empty directory that will be used to store the numerical derivatives of the theory with respect to each parameter.
  4. `fisher_matrices`: An empty directory that can be used to store any Fisher matrices that are computed from the derivatives in the `derivs` directory.
  
**Note**: you should generally pass a new `fisher_dir` if you change any of the *optional* arguments passed to `Fisher` (described below); the expection is the `use_H0` flag. Unless you pass `overwrite=True`, the saved fiducial parameter and step size files will be loaded in when you initialize `Fisher` a second time with the same `fisher_dir`, and any saved numerical derivatives will not be re-calculated; if you, e.g., changed the set of fiducial parameters, this change would not be reflected in the calculated Fisher matrices.
  
Below, we set the `fisher_dir` to be a directory (relative to this notebook) named `example_output/hd_example` (or `example_output/hd_example_CLASS` if you are using CLASS by setting `use_class=True` below).

The following arguments are _optional_. You may use CLASS instead of CAMB by setting `use_class=True` below; we use the default values for all other arguments in this example:

- `use_class` (default=`False`): You may pass `use_class=True` to use CLASS instead of CAMB for the theory calculations; by default, CAMB is used. 
  
  **If you are using CLASS instead of CAMB**:
  - **CLASS must be modified** when using the default CLASS parameters in `hdfisher`; see the "README" file for more information and modification instructions.
  - Note that CLASS does not calculate delensed power spectra.

- `fiducial_params` (default=`None`): You may pass either a dictionary of parameter names and their fiducial values, the path to a YAML file where such a dictionary has been saved, or leave `fiducial_params=None` to use the default set of parameters. 
  
  All parameters (cosmology, accuracy, etc.) in `fiducial_params` are passed to CAMB or CLASS, so the parameter names should be valid CAMB or CLASS parameter names. The expections are three parameter "aliases" we define within `hdfisher`:
  - For CAMB, we define (and vary, by default):
    - `theta`, which is $100 \theta_\mathrm{MC}$ (i.e., `theta = 100 * cosmomc_theta`)
    - `logA`, which is $\ln(10^{10} A_\mathrm{s})$
  - For CLASS, we define (and vary, by default) `sum_m_ncdm` for the sum of the neutrino masses (in eV), which is calculated based on the values of `m_ncdm`, `N_ncdm`, and `deg_ncdm` in `fiducial_params`, if applicable.

- `step_sizes` (default=`None`): You may pass either a dictionary containing parameter names and information about the step size to use when varying each, the path to a YAML file where such a dictionary has been saved, or leave `step_sizes=None` to use the default set of step sizes. 
  
  Parameters in `step_sizes` that do not have a corresponding fiducial value given in `fiducial_params` will be ignored.
  
  For example, to vary $n_\mathrm{s}$ by $\pm$ 1% of its fiducial value of 0.9649, the `step_sizes` file should contain one of the following blocks:
    ```yaml
    ns:
      step_size: 0.01
      step_type: relative
    ```
    
    
    or
    
    ```yaml
    ns:
      step_size: 0.009649
      step_type: absolute
    
    
    ```
    
    
    or, if `step_sizes` is a dictionary (using the first example for a relative step size),
    
    ```python
    step_sizes = {'ns': {'step_size': 0.01, 'step_type': 'relative'}, 
                  # other parameters 
                 }
    ```

- `fisher_params` (default=`None`): A list of names of varied parameters (each must have a key in both `fiducial_params` and `step_sizes`), or left as `None` to vary all possible parameters (i.e., those with both a fiducial value and a step size, with one exception; see `use_H0` below).

- `use_H0` (default = `False`): This is used when `fiducial_params` and `step_sizes` each have keys for _both_ the Hubble constant (`'H0'`, or `'h'` in CLASS) and the angular size of the sound horizon at last scattering (`cosmomc_theta` (or `theta`) in CAMB, `theta_s_100` in CLASS): only one of these can be passed to CAMB or CLASS, and therefore only one can be included in the set of varied parameters. If `use_H0 = False`, the default set of varied parameters includes the angular size of the sound horizon at last scattering; if `use_H0=True`, it contains the Hubble constant instead.
  
  Note that the `use_H0` flag passed to `Fisher` sets the *default* list of varied parameters; the methods of the `Fisher` class have a `use_H0` flag and/or accept a list of parameters (when applicable), which may be passed to override these defaults.

- `feedback` (default=`False`): Only used to set the defaults if the `fiducial_params` and/or `step_sizes` aren't given. If `True`, the HMCode2020 baryonic feedback model is used instead of the HMCode2016 CDM-only model, and the parameter that characterizes the strength of the baryonic feedback (`'HMCode_logT_AGN'` in CAMB, `'log10T_heat_hmcode'` in CLASS) will be varied by default.

- `overwrite` (default=`False`): Pass `overwrite=True` to overwrite any existing files (including the fiducial parameters, step sizes, numerical derivatives) in the `fisher_dir` and its sub-directories. Otherwise, these files will be loaded instead of re-calculated.

We also provide the following extra optional arguments:

- `exp` (default=`'hd'`): Used to load the covariance matrix for the mock data spectra, and determine the multipole ranges used in the calculation. Must be `'hd'`, `'s4'`, or `'so'` for CMB-HD-like, CMB-S4-like, or SO-like configurations, respectively. 
- `hd_data_version` (default=`'latest'`): The version number used to load the CMB-HD mock data from the `hdMockData` [repository](https://github.com/CMB-HD/hdMockData); please see that repository for more information about the data versions. By default, the latest CMB-HD data is used.
    - If you are running new forecasts, please use the latest CMB-HD data.
    - We provide this option to allow you to reproduce the results of MacInnis et. al. (2023), which used version `'v1.0'`.
- `pol_only_lensing` (default=`False`): Only for version `'v1.2'` of the mock CMB-HD data. If `True`, only polarization lensing estimators (*EE* and *EB*) are used in the calculation of the lensing noise; by default, the *TT*, *TE*, *TB*, *EE*, and *EB* estimators are used.
- `hd_lmax` (default=`None`): Only for version `'v1.0'` of the mock CMB-HD delensed data. This can be an integer value lower than the baseline of $\ell_{max}, L_{max} = 20,100$, which is used by default if `hd_lmax=None`. The options are `1000`, `3000`, `5000`, and `10000`.
- `include_fg` (default=`True`): Only for version `'v1.0'` of the mock CMB-HD lensed data (i.e., no delensing). Set `include_fg=False` to calculate a Fisher matrix without including the effects of residual extragalactic foregrounds in temperature. Note that later versions of the CMB-HD mock data always include these effects.

In the cell below, you may set `use_class=True` to use CLASS instead of CAMB. You may also change the `fisher_dir` if you wish. In the last line of the cell, we initialize the `Fisher` class.

In [2]:
# set `use_class=True` to use CLASS instead of CAMB:
use_class = False

# here, we set a default `fisher_dir` that holds the output:
fisher_root = 'hd_example_CLASS' if use_class else 'hd_example'
output_dir = os.path.join(os.getcwd(), 'example_output')
fisher_dir = os.path.join(output_dir, fisher_root)

# initialize the `Fisher` class:
fisherlib = fisher.Fisher(fisher_dir)

# Calculating new Fisher derivatives

Before calculating new Fisher matrices, you must calculate new Fisher derivatives first. This is done using the `calculate_fisher_derivs` method of the `Fisher` class initialized above. 


**Note** that this requires many calls to CAMB or CLASS, using the default high-accuracy settings of `hdfisher`; we recommend that you submit the calculation of the derivatives as a job on a cluster (as opposed to running on a login node).


There are two ways to do the calculation:

- With MPI: You can calculate the derivatives in parallel using the Python script `example_calculate_fisher_derivs.py`.  For example, to use 4 parallel MPI processes, run one of the following commands (or the equivalent command for your machine):
  
  **If you are using CAMB** (i.e. `use_class=False`, the default):
  
  ```bash
    mpirun -np 4 python example_calculate_fisher_derivs.py
  
  
  ```
  
  **If you are using CLASS** (i.e. `use_class=True`):
  
  ```bash
    mpirun -np 4 python example_calculate_fisher_derivs.py --use_class
  
  
  ```



- Without MPI: You can either use the Python script `example_calculate_fisher_derivs.py` (recommended), or un-comment and run the code in the following cell (see the note above). This may take significantly longer than the first option. To use the Python script, run one of the following commands:
  
  **If you are using CAMB** (i.e. `use_class=False`, the default):
  
  ```bash
    python example_calculate_fisher_derivs.py
  
  
  ```
  
  **If you are using CLASS** (i.e. `use_class=True`):
  
  ```bash
    python example_calculate_fisher_derivs.py --use_class
  
  
  ```


After the derivatives have been calculated and saved, you may proceed to the next section.

In [3]:
#fisherlib.calculate_fisher_derivs()

# Calculating new Fisher matrices

Once you've calculated the derivatives of the CMB and BAO theory with respect to parameters, you can calculate Fisher matrices for any set of those parameters. This is done using the `get_fisher` method of the `Fisher` class. 

The `get_fisher` method returns a tuple with two items: the first is the Fisher matrix (as a two-dimensional `numpy` array), and the second is a list of parameter names corresponding to the parameter rows and columns of the Fisher matrix.

The (optional) argumemts for `Fisher.get_fisher` are:
- `cmb_type` (default = `delensed`): Pass `cmb_type='delensed'` to calculate a Fisher matrix using delensed CMB power spectra, or `cmb_type='lensed'` for lensed CMB power spectra. **Note** that you must *always* pass `cmb_type='lensed'` when `use_class=True`.
- `params` (default = `None`): This is an optional list of names of parameter to include in the Fisher matrix. By default, all parameters that you have calculated derivatives for will be used; however, see the note below about $H_0$ and $100\theta_\mathrm{MC}$ (in CAMB) or $100\theta_*$ (in CLASS).
- `priors` (default = `None`): You may pass a dictionary with key, value pairs `param: prior_width`, where `param` is the parameter name and `prior_width` is the width of the Gaussian prior to apply; or you may pass `priors=True` to apply the default CMB-HD priors (a prior of $\sigma(\tau) = 0.005$, and a 0.06% prior on the baryonic feedback parameter, if applicable). If `priors=None` or `priors=False`, no priors are applied.
- `use_H0` (default = `None`): Used to override the value of `use_H0` passed when initializing the `Fisher` class.
- `with_desi` (default = `False`): Whether to include mock DESI BAO, by adding its Fisher matrix to the CMB Fisher matrix being calculated.
- `save` (default = `False`), `fname` (default = `None`), and `overwrite` (default is `None`): By default, the Fisher matrix will not be saved. Pass `save=True` to save the Fisher matrix to the `fisher_matrices` subdirectory of your `fisher_dir` directory. If `save=True`, you must also pass a file name to `fname` (relative to the `fisher_matrices` directory); if the file exists and `overwrite=False`, an error will be raised. By default, the value of `overwrite` passed to the `Fisher` class is used; you may override it by passing `overwrite` to `get_fisher`.


If you're only interested in the parameter uncertainties, you can instead use the `get_fisher_errors` method, which takes the same arguments as `get_fisher`.

In [4]:
use_H0 = False
cmb_type = 'lensed' if fisherlib.use_class else 'delensed'
errs = fisherlib.get_fisher_errors(cmb_type, priors=True, use_H0=use_H0)
errs_with_desi = fisherlib.get_fisher_errors(cmb_type, priors=True, use_H0=use_H0,  with_desi=True)

# load pre-computed results for comparison:
hd_fmat, params = fisherlib.load_example_hd_fisher(cmb_type=cmb_type, use_H0=use_H0, with_desi=False)
hd_errs = fisher.get_fisher_errors(hd_fmat, params)
    
hd_desi_fmat, params = fisherlib.load_example_hd_fisher(cmb_type=cmb_type, use_H0=use_H0, with_desi=True)
hd_desi_errs = fisher.get_fisher_errors(hd_desi_fmat, params)

Define some functions to print out the results:

In [5]:
# order of parameters in tables
param_list = ['ombh2', 'omch2', 'logA', 'ns', 'tau', 'H0', 'nnu', 'mnu']

def ratio(dict1, dict2, fill_value=None):
    """Returns a dictionary with the ratio of the values in `dict1` to those 
    in `dict2` for each key they share in common. If the keys don't match, you
    can pass a `fill_value` as a placeholder for any keys not in both dictionaries;
    otherwise leave it as `None` to exclude those keys from the returned dictionary."""
    rdict = {}
    keys1 = list(dict1.keys())
    keys2 = list(dict2.keys())
    all_keys = list(set(keys1 + keys2))
    for key in all_keys:
        if (key in dict1) and (key in dict2):
            rdict[key] = dict1[key] / dict2[key]
        elif fill_value is not None:
            rdict[key] = fill_value
    return rdict


def print_table(list_of_dicts, list_of_labels, title=None, list_of_keys=None):
    """
    Display a table of keys and values held in each dictionary in the `list_of_dicts`.
    
    Parameters
    ----------
    list_of_dicts : list of dict
        A list of dictionaries.
    list_of_labels : list of str
        A list holding a label for each dictionary in `list_of_dicts`, in
        the same order that they appear in `list_of_dicts`.
    title : str or None, default=None
        A title that is printed out above the table.
    list_of_keys : list or None, default=None
        A list of keys to use in the table. The table will only contain 
        values corresponding to the keys in `list_of_keys`, and in the same
        order. If `None`, all keys are used. In either case, the dictionaries
        in `list_of_dicts` do not need to contain the same set of keys.
    
    """
    table = pd.DataFrame(list_of_dicts, index=list_of_labels, columns=list_of_keys)
    table = table.T
    if title is not None:
        print('\n', title) 
    display(table)

Print out a table comparing the new results to the pre-computed baseline case:

In [6]:
print('\nComparison of new Fisher forecasts with the precomputed baseline case:')

table_list = [hd_desi_errs, errs_with_desi, ratio(errs_with_desi, hd_desi_errs)]
label_list = [f'Precomputed HD {cmb_type} + DESI', 
              f'New HD {cmb_type} + DESI forecast', 'New / Precomputed ratio']
print_table(table_list, label_list)

table_list = [hd_errs, errs, ratio(errs, hd_errs)]
label_list = [f'Precomputed HD {cmb_type}', 
              f'New HD {cmb_type} forecast', 'New / Precomputed ratio']
print_table(table_list, label_list)


Comparison of new Fisher forecasts with the precomputed baseline case:


,Precomputed HD delensed + DESI,New HD delensed + DESI forecast,New / Precomputed ratio
logA,0.007867,0.007867,1.0
mnu,0.026153,0.026153,1.0
nnu,0.014160,0.014160,1.0
ns,0.001527,0.001527,1.0
ombh2,0.000025,0.000025,1.0
omch2,0.000366,0.000366,1.0
tau,0.004207,0.004207,1.0
theta,0.000060,0.000060,1.0


,Precomputed HD delensed,New HD delensed forecast,New / Precomputed ratio
logA,0.008424,0.008424,1.0
mnu,0.051286,0.051286,1.0
nnu,0.016791,0.016791,1.0
ns,0.002430,0.002430,1.0
ombh2,0.000026,0.000026,1.0
omch2,0.000489,0.000489,1.0
tau,0.004308,0.004308,1.0
theta,0.000078,0.000078,1.0
